# Bronze Layer - Hospital Patient Analytics Pipeline


## Define Source File Paths


In [0]:
patient_path = "/Volumes/workspace/default/final_assignment/raw_data/patients.csv"
encounter_path = "/Volumes/workspace/default/final_assignment/raw_data/encounters.csv"
diagnosis_path = "/Volumes/workspace/default/final_assignment/raw_data/diagnoses.csv"
claims_path = "/Volumes/workspace/default/final_assignment/raw_data/claims_and_billing.csv"
provider_path = "/Volumes/workspace/default/final_assignment/raw_data/providers.csv"

## Load Patients Data


In [0]:
from pyspark.sql.functions import current_timestamp

patients_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(patient_path)
    .select(
        "patient_id",
        "first_name",
        "last_name",
        "age",
        "city"
    )
    .withColumn("load_ts", current_timestamp())
)

## Validate Patients Data

In [0]:
patients_df.printSchema()
print(patients_df.count())
display(patients_df.limit(5))

root
 |-- patient_id: string (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- city: string (nullable = true)
 |-- load_ts: timestamp (nullable = false)

60000


patient_id,first_name,last_name,age,city,load_ts
PAT000001,Danielle,Johnson,85,Bakersfield,2026-06-23T19:32:03.329Z
PAT000002,Anna,Baldwin,15,Bakersfield,2026-06-23T19:32:03.329Z
PAT000003,James,Jones,4,Portland,2026-06-23T19:32:03.329Z
PAT000004,Veronica,Bowman,48,Seattle,2026-06-23T19:32:03.329Z
PAT000005,Carl,Gentry,83,null,2026-06-23T19:32:03.329Z


## Load Visit Data

In [0]:
encounters_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(encounter_path)
    .select(
        "encounter_id",
        "patient_id",
        "provider_id",
        "visit_date"
    )
    .withColumn("load_ts", current_timestamp())
)

## Validate Visit Data

In [0]:
encounters_df.printSchema()
print(encounters_df.count())
display(encounters_df.limit(5))

root
 |-- encounter_id: string (nullable = true)
 |-- patient_id: string (nullable = true)
 |-- provider_id: string (nullable = true)
 |-- visit_date: date (nullable = true)
 |-- load_ts: timestamp (nullable = false)

70000


encounter_id,patient_id,provider_id,visit_date,load_ts
ENC017535,PAT000001,PRO00865,2025-01-03,2026-06-23T19:32:17.882Z
ENC030367,PAT000001,PRO00579,2025-03-30,2026-06-23T19:32:17.882Z
ENC038533,PAT000002,PRO01227,2025-02-26,2026-06-23T19:32:17.882Z
ENC056041,PAT000003,PRO00055,2025-01-27,2026-06-23T19:32:17.882Z
ENC046849,PAT000004,PRO00043,2025-01-26,2026-06-23T19:32:17.882Z


## Load diagnoses Data

In [0]:
diagnoses_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(diagnosis_path)
    .select(
        "encounter_id",
        "diagnosis_code"
    )
    .withColumn("load_ts", current_timestamp())
)

## Validate diagnoses Data

In [0]:
diagnoses_df.printSchema()
print(diagnoses_df.count())
display(diagnoses_df.limit(5))

root
 |-- encounter_id: string (nullable = true)
 |-- diagnosis_code: string (nullable = true)
 |-- load_ts: timestamp (nullable = false)

70000


encounter_id,diagnosis_code,load_ts
ENC000001,J45.909,2026-06-23T19:32:23.208Z
ENC000002,R06.02,2026-06-23T19:32:23.208Z
ENC000003,N20.0,2026-06-23T19:32:23.208Z
ENC000004,R07.9,2026-06-23T19:32:23.208Z
ENC000005,K58.9,2026-06-23T19:32:23.208Z


## Load Bill Data

In [0]:
claims_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(claims_path)
    .select(
        "encounter_id",
        "billed_amount"
    )
    .withColumn("load_ts", current_timestamp())
)

## Validate Bill Data

In [0]:
claims_df.printSchema()
print(claims_df.count())
display(claims_df.limit(5))

root
 |-- encounter_id: string (nullable = true)
 |-- billed_amount: double (nullable = true)
 |-- load_ts: timestamp (nullable = false)

70000


encounter_id,billed_amount,load_ts
ENC000001,1971.52,2026-06-23T19:32:29.399Z
ENC000002,1243.8,2026-06-23T19:32:29.399Z
ENC000003,4854.11,2026-06-23T19:32:29.399Z
ENC000004,2638.21,2026-06-23T19:32:29.399Z
ENC000005,1046.99,2026-06-23T19:32:29.399Z


## Load Doctor Data

In [0]:
providers_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(provider_path)
    .select(
        "provider_id",
        "name",
        "specialty",
        "department"
    )
    .withColumn("load_ts", current_timestamp())
)

## Validate Doctor Data

In [0]:
providers_df.printSchema()
print(providers_df.count())
display(providers_df.limit(5))

root
 |-- provider_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- specialty: string (nullable = true)
 |-- department: string (nullable = true)
 |-- load_ts: timestamp (nullable = false)

1491


provider_id,name,specialty,department,load_ts
PRO00001,Michael Hernandez,General Practitioner,Emergency Department,2026-06-23T19:32:33.584Z
PRO00002,Dawn Owens,General Practitioner,Emergency Department,2026-06-23T19:32:33.584Z
PRO00003,Catherine Knight,General Practitioner,Emergency Department,2026-06-23T19:32:33.584Z
PRO00004,Roy Williams,General Practitioner,Emergency Department,2026-06-23T19:32:33.584Z
PRO00005,Cheryl Norris,General Practitioner,Emergency Department,2026-06-23T19:32:33.584Z


## Bronze Delta Tables

> patient bronze Delta Tables

In [0]:
patients_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save("/Volumes/workspace/default/final_assignment/bronze/patients")

> Visit bronze Delta Tables

In [0]:
encounters_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save("/Volumes/workspace/default/final_assignment/bronze/encounters")

> diagnoses Delta Tables

In [0]:
diagnoses_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save("/Volumes/workspace/default/final_assignment/bronze/diagnoses")

> Bill Delta Tables

In [0]:
claims_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save("/Volumes/workspace/default/final_assignment/bronze/claims")

> Doctor Delta Table

In [0]:
providers_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save("/Volumes/workspace/default/final_assignment/bronze/providers")

## Verify Bronze Layer Tables

In [0]:
%sql
SHOW TABLES IN workspace.default;

database,tableName,isTemporary
default,messy_customers_large,false


In [0]:
encounters_df.printSchema()
diagnoses_df.printSchema()
claims_df.printSchema()
providers_df.printSchema()

root
 |-- encounter_id: string (nullable = true)
 |-- patient_id: string (nullable = true)
 |-- provider_id: string (nullable = true)
 |-- visit_date: date (nullable = true)
 |-- load_ts: timestamp (nullable = false)

root
 |-- encounter_id: string (nullable = true)
 |-- diagnosis_code: string (nullable = true)
 |-- load_ts: timestamp (nullable = false)

root
 |-- encounter_id: string (nullable = true)
 |-- billed_amount: double (nullable = true)
 |-- load_ts: timestamp (nullable = false)

root
 |-- provider_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- specialty: string (nullable = true)
 |-- department: string (nullable = true)
 |-- load_ts: timestamp (nullable = false)



## Verify Bronze Layer Tables

In [0]:
# Verify Patients Bronze
display(
    spark.read.format("delta")
    .load("/Volumes/workspace/default/final_assignment/bronze/patients").limit(5)
)

# Verify Encounters Bronze
display(
    spark.read.format("delta")
    .load("/Volumes/workspace/default/final_assignment/bronze/encounters").limit(5)
)

# Verify Diagnoses Bronze
display(
    spark.read.format("delta")
    .load("/Volumes/workspace/default/final_assignment/bronze/diagnoses").limit(5)
)

# Verify Claims Bronze
display(
    spark.read.format("delta")
    .load("/Volumes/workspace/default/final_assignment/bronze/claims").limit(5)
)

# Verify Providers Bronze
display(
    spark.read.format("delta")
    .load("/Volumes/workspace/default/final_assignment/bronze/providers").limit(5)
)

patient_id,first_name,last_name,age,city,load_ts
PAT000001,Danielle,Johnson,85,Bakersfield,2026-06-23T19:32:36.705Z
PAT000002,Anna,Baldwin,15,Bakersfield,2026-06-23T19:32:36.705Z
PAT000003,James,Jones,4,Portland,2026-06-23T19:32:36.705Z
PAT000004,Veronica,Bowman,48,Seattle,2026-06-23T19:32:36.705Z
PAT000005,Carl,Gentry,83,null,2026-06-23T19:32:36.705Z


encounter_id,patient_id,provider_id,visit_date,load_ts
ENC017535,PAT000001,PRO00865,2025-01-03,2026-06-23T19:32:41.529Z
ENC030367,PAT000001,PRO00579,2025-03-30,2026-06-23T19:32:41.529Z
ENC038533,PAT000002,PRO01227,2025-02-26,2026-06-23T19:32:41.529Z
ENC056041,PAT000003,PRO00055,2025-01-27,2026-06-23T19:32:41.529Z
ENC046849,PAT000004,PRO00043,2025-01-26,2026-06-23T19:32:41.529Z


encounter_id,diagnosis_code,load_ts
ENC000001,J45.909,2026-06-23T19:32:44.614Z
ENC000002,R06.02,2026-06-23T19:32:44.614Z
ENC000003,N20.0,2026-06-23T19:32:44.614Z
ENC000004,R07.9,2026-06-23T19:32:44.614Z
ENC000005,K58.9,2026-06-23T19:32:44.614Z


encounter_id,billed_amount,load_ts
ENC000001,1971.52,2026-06-23T19:32:47.544Z
ENC000002,1243.8,2026-06-23T19:32:47.544Z
ENC000003,4854.11,2026-06-23T19:32:47.544Z
ENC000004,2638.21,2026-06-23T19:32:47.544Z
ENC000005,1046.99,2026-06-23T19:32:47.544Z


provider_id,name,specialty,department,load_ts
PRO00001,Michael Hernandez,General Practitioner,Emergency Department,2026-06-23T19:32:50.530Z
PRO00002,Dawn Owens,General Practitioner,Emergency Department,2026-06-23T19:32:50.530Z
PRO00003,Catherine Knight,General Practitioner,Emergency Department,2026-06-23T19:32:50.530Z
PRO00004,Roy Williams,General Practitioner,Emergency Department,2026-06-23T19:32:50.530Z
PRO00005,Cheryl Norris,General Practitioner,Emergency Department,2026-06-23T19:32:50.530Z
